In [1]:
## Running the Risk Metric Calculations
%run risk_metric_calc.ipynb

<class 'pandas.DataFrame'>
DatetimeIndex: 1343 entries, 2021-01-04 to 2026-05-08
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BUFR    1343 non-null   float64
 1   IJR     1343 non-null   float64
 2   SMH     1343 non-null   float64
 3   SPY     1343 non-null   float64
 4   UGL     1343 non-null   float64
 5   VO      1343 non-null   float64
dtypes: float64(6)
memory usage: 73.4 KB


In [2]:
## Resetting Individual Indexes for the Tables and Copying Them
portfolio_returns_export = portfolio_returns.reset_index()
monthly_returns_export = monthly_returns.copy()
summary_metrics_export = summary_metrics.copy()
prices_long_export = prices_long.copy()
returns_long_export = returns_long.copy()
weights_export = weights_df.copy()
contribution_export = contribution_summary.copy()

## Saving to SQLlite Function
def save_to_sqlite(dataframes_dict, db_path):
    conn = sqlite3.connect(db_path)
    
    for table_name, df in dataframes_dict.items():
        df.to_sql(table_name, conn, if_exists="replace", index=False)
    
    conn.close()
    
## Note. Creating Buckets for the Years and Months on the Table
portfolio_returns_export['Date'] = pd.to_datetime(portfolio_returns_export['Date'])
portfolio_returns_export['Year'] = portfolio_returns_export['Date'].dt.year
portfolio_returns_export['Month'] = portfolio_returns_export['Date'].dt.strftime('%Y-%m-01')

In [3]:
## List of Tables to Save
tables_to_save = {
    "fact_prices": prices_long_export,
    "fact_returns": returns_long_export,
    "fact_portfolio_returns": portfolio_returns_export,
    "fact_monthly_returns": monthly_returns_export,
    "fact_summary_metrics": summary_metrics_export,
    "dim_weights": weights_export,
    "fact_contribution_summary": contribution_export
}

## Saving the above tables to SQLite
save_to_sqlite(tables_to_save, DB_PATH)

In [4]:
## ---------------------------------------------------------
## Running Queries on the Database and Saving Them as Tables
## ---------------------------------------------------------

conn = sqlite3.connect(DB_PATH)

In [5]:
## Querying the Portfolio Performance Growth by Year
query = """
DROP TABLE IF EXISTS analysis_portfolio_growth;

CREATE TABLE analysis_portfolio_growth AS
SELECT 
    STRFTIME('%Y', date) AS year,
    EXP(SUM(LOG(1 + portfolio_return))) - 1 AS annual_return
FROM fact_portfolio_returns
GROUP BY year
ORDER BY year;
"""
conn.executescript(query)
conn.commit()

## Checking our Results
query = "SELECT * FROM analysis_portfolio_growth"
pd.read_sql(query, conn)

,year,annual_return
0,2021,0.077023
1,2022,-0.071844
2,2023,0.108263
3,2024,0.107049
4,2025,0.152299
5,2026,0.071701


In [6]:
## Querying the Difference Between Portfolio and Benchmark Returns
query = """
DROP TABLE IF EXISTS analysis_monthly_excess;

CREATE TABLE analysis_monthly_excess AS
SELECT 
    date, 
    Portfolio_Monthly_Return, 
    Benchmark_Monthly_Return,
    (Portfolio_Monthly_Return - Benchmark_Monthly_Return) AS excess_return
FROM fact_monthly_returns
ORDER BY date;
"""
conn.executescript(query)
conn.commit()

## Checking our Results
query = "SELECT * FROM analysis_monthly_excess"

pd.read_sql(query, conn)

,Date,Portfolio_Monthly_Return,Benchmark_Monthly_Return,excess_return
0,2021-01-31 00:00:00,-0.003820,0.003471,-0.007290
1,2021-02-28 00:00:00,0.010984,0.027806,-0.016821
2,2021-03-31 00:00:00,0.018457,0.045400,-0.026943
3,2021-04-30 00:00:00,0.036910,0.052910,-0.016000
4,2021-05-31 00:00:00,0.041343,0.006566,0.034777
...,...,...,...,...
60,2026-01-31 00:00:00,0.082143,0.014738,0.067405
61,2026-02-28 00:00:00,0.045891,-0.008642,0.054533
62,2026-03-31 00:00:00,-0.081504,-0.049380,-0.032125
63,2026-04-30 00:00:00,0.092454,0.105053,-0.012599


In [7]:
## Querying the Cumulative Return Utilizing a Different Formula
query = """
DROP TABLE IF EXISTS analysis_cumulative_return;

CREATE TABLE analysis_cumulative_return AS
SELECT 
    date,
    portfolio_return,
    EXP(SUM(LOG(1 + portfolio_return)) OVER (ORDER BY date)) - 1 AS cumulative_return
FROM fact_portfolio_returns
ORDER BY date;
"""
conn.executescript(query)
conn.commit()

## Checking our Results
query = "SELECT * FROM analysis_cumulative_return"

pd.read_sql(query, conn)

,Date,Portfolio_Return,cumulative_return
0,2021-01-05 00:00:00,0.010931,0.004733
1,2021-01-06 00:00:00,0.005506,0.007131
2,2021-01-07 00:00:00,0.011831,0.012289
3,2021-01-08 00:00:00,-0.014172,0.006033
4,2021-01-11 00:00:00,0.000735,0.006354
...,...,...,...
1337,2026-05-04 00:00:00,-0.011738,0.486582
1338,2026-05-05 00:00:00,0.013428,0.495218
1339,2026-05-06 00:00:00,0.025081,0.511391
1340,2026-05-07 00:00:00,-0.005866,0.507534


In [8]:
## Querying the Worst Drawdown Dates as a Table
query = """
DROP TABLE IF EXISTS analysis_worst_drawdowns;

CREATE TABLE analysis_worst_drawdowns AS
SELECT 
    date,
    portfolio_drawdown
    FROM fact_portfolio_returns
    ORDER BY portfolio_drawdown ASC
    LIMIT 10;
"""
conn.executescript(query)
conn.commit()

## Checking our Results
query = """
SELECT * FROM analysis_worst_drawdowns
"""

pd.read_sql(query, conn)

,Date,Portfolio_Drawdown
0,2022-10-14 00:00:00,-0.255219
1,2022-10-12 00:00:00,-0.247481
2,2022-10-11 00:00:00,-0.246187
3,2022-10-20 00:00:00,-0.246091
4,2022-09-30 00:00:00,-0.245897
5,2022-09-26 00:00:00,-0.245223
6,2022-09-27 00:00:00,-0.244644
7,2022-10-19 00:00:00,-0.241917
8,2022-10-10 00:00:00,-0.240263
9,2022-10-17 00:00:00,-0.239931


In [9]:
## Querying the Negative Days as a Table
query = """
DROP TABLE IF EXISTS analysis_negative_days;

CREATE TABLE analysis_negative_days AS
SELECT 
    COUNT(*) AS total_days,
    SUM(CASE WHEN portfolio_return > 0 THEN 1 ELSE 0 END) AS postitive_days,
    SUM(CASE WHEN portfolio_return < 0 THEN 1 ELSE 0 END) AS negative_days,
    1.0 * SUM(CASE WHEN portfolio_return > 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_positive_days, 
    1.0 * SUM(CASE WHEN portfolio_return < 0 THEN 1 ELSE 0 END) / COUNT(*) AS pct_negative_days
FROM fact_portfolio_returns;
"""

conn.executescript(query)
conn.commit()

query = """
SELECT * FROM analysis_negative_days
"""
pd.read_sql(query, conn)

,total_days,postitive_days,negative_days,pct_positive_days,pct_negative_days
0,1342,734,608,0.546945,0.453055


In [10]:
## Querying an Annual Return Table for Further Analysis
query = """
SELECT date, portfolio_return, benchmark_return FROM fact_portfolio_returns;
"""

portfolio_returns_export = pd.read_sql(query, conn)

portfolio_returns_export["Date"] = pd.to_datetime(portfolio_returns_export["Date"])
portfolio_returns_export["year"] = portfolio_returns_export["Date"].dt.year

annual_returns = (
    portfolio_returns_export
    .groupby("year")
    .agg(
        portfolio_annual_return=("Portfolio_Return", lambda x: (1 + x).prod() - 1),
        benchmark_annual_return=("Benchmark_Return", lambda x: (1 + x).prod() - 1)
    )
    .reset_index()
)

annual_returns["excess_annual_return"] = (
    annual_returns["portfolio_annual_return"] 
    - annual_returns["benchmark_annual_return"]
)

annual_returns.to_sql(
    "analysis_annual_returns",
    conn,
    if_exists="replace",
    index=False
)

query = """
SELECT * FROM analysis_annual_returns
"""
pd.read_sql(query, conn)

,year,portfolio_annual_return,benchmark_annual_return,excess_annual_return
0,2021,0.186318,0.305054,-0.118737
1,2022,-0.157743,-0.181753,0.024010
2,2023,0.267050,0.261758,0.005293
3,2024,0.263858,0.248865,0.014993
4,2025,0.385987,0.177191,0.208797
5,2026,0.172863,0.084635,0.088228


In [11]:
## Querying Best Performing Months
query = """
DROP TABLE IF EXISTS analysis_contribution_summary;

CREATE TABLE analysis_contribution_summary AS
SELECT 
    date,
    portfolio_monthly_return,
    RANK() OVER (ORDER BY portfolio_monthly_return DESC) AS performance_rank
FROM fact_monthly_returns;
"""
conn.executescript(query)
conn.commit()

query = """
SELECT * FROM analysis_contribution_summary
"""
pd.read_sql(query, conn)

,Date,Portfolio_Monthly_Return,performance_rank
0,2022-11-30 00:00:00,0.096564,1
1,2023-01-31 00:00:00,0.094035,2
2,2026-04-30 00:00:00,0.092454,3
3,2023-11-30 00:00:00,0.088638,4
4,2026-01-31 00:00:00,0.082143,5
...,...,...,...
60,2023-09-30 00:00:00,-0.061691,61
61,2022-04-30 00:00:00,-0.080932,62
62,2026-03-31 00:00:00,-0.081504,63
63,2022-06-30 00:00:00,-0.082391,64


In [12]:
## Exporting to Power Bi as a .csv file
tables = [
    "fact_returns",
    "fact_portfolio_returns",
    "fact_monthly_returns",
    "fact_summary_metrics",
    "fact_contribution_summary",
    "dim_weights",
    "analysis_portfolio_growth",
    "analysis_monthly_excess",
    "analysis_cumulative_return",
    "analysis_worst_drawdowns",
    "analysis_negative_days",
    "analysis_annual_returns",
    "analysis_contribution_summary",
]


OUTPUT_DIR = "tables_csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    df.to_csv(f"tables_csv/{table}.csv", index=False)


In [13]:
conn.close()